# Penerapan Metode IndoBERT untuk Mengukur Kemiripan Berdasarkan Judul Konten pada Sistem Rekomendasi Channel YouTube

**Metodologi:** Content-Based Filtering menggunakan IndoBERT Fine-Tuned + Cosine Similarity

**Alur Pipeline:**
1. Install & Import Library
2. Load Dataset JSON
3. Text Cleaning
4. Case Folding
5. Tokenisasi dengan Stanza (Bahasa Indonesia)
6. Split Data Terintegrasi 70/10/20 (Train/Validation/Test Channel)
7. Konfigurasi Model IndoBERT
8. Fine-Tuning IndoBERT (Supervised Classification)
9. Generate Embedding Judul Video (Fine-Tuned Encoder)
10. Agregasi Vektor Channel (Mean Pooling + L2 Normalization)
11. Cosine Similarity Matrix & Fungsi Rekomendasi
12. Evaluasi Sistem (Precision@K pada Query Set/Test Channel)
13. Simpan Artefak Final Fine-Tuning

---
## 1. Install & Import Library

In [1]:
# Install library yang diperlukan
%pip install torch transformers stanza scikit-learn pandas numpy tqdm --quiet


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import re
import os
import numpy as np
import pandas as pd

# Deep Learning & NLP
import torch
from transformers import AutoTokenizer

# Stanza untuk Tokenisasi Bahasa Indonesia
import stanza

# Similarity & Evaluation
from sklearn.metrics.pairwise import cosine_similarity

# Progress bar
from tqdm import tqdm

# Konfigurasi device (GPU jika tersedia, fallback ke CPU jika tidak kompatibel)
def get_device():
    """Deteksi device dengan fallback ke CPU jika CUDA tidak kompatibel."""
    if not torch.cuda.is_available():
        return torch.device('cpu')

    try:
        major, minor = torch.cuda.get_device_capability(0)
        # PyTorch build yang terpasang di environment ini hanya mendukung CC >= 7.5
        if (major, minor) < (7, 5):
            print(
                f"⚠️  GPU {torch.cuda.get_device_name(0)} memiliki compute capability "
                f"sm_{major}{minor}, jadi fallback ke CPU"
            )
            return torch.device('cpu')

        # Test ringan agar benar-benar memastikan CUDA bisa dipakai
        _ = torch.randn(1, device='cuda')
        return torch.device('cuda')
    except Exception as e:
        print(f"⚠️  CUDA tidak bisa dipakai ({e}) → fallback ke CPU")
        return torch.device('cpu')

device = get_device()
print(f'Menggunakan device  : {device}')
print(f'PyTorch version     : {torch.__version__}')
print('Import library selesai.')

/home/candimadam/Documents/Tugas Akhir/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


⚠️  GPU NVIDIA GeForce MX350 memiliki compute capability sm_61, jadi fallback ke CPU
Menggunakan device  : cpu
PyTorch version     : 2.11.0+cu130
Import library selesai.


/home/candimadam/Documents/Tugas Akhir/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:371: UserWarning: Found GPU0 NVIDIA GeForce MX350 which is of compute capability (CC) 6.1.
The following list shows the CCs this version of PyTorch was built for and the hardware CCs it supports:
- 7.5 which supports hardware CC >=7.5,<8.0
- 8.0 which supports hardware CC >=8.0,<9.0 except {8.7}
- 8.6 which supports hardware CC >=8.6,<9.0 except {8.7}
- 9.0 which supports hardware CC >=9.0,<10.0
- 10.0 which supports hardware CC >=10.0,<11.0 except {10.1}
- 12.0 which supports hardware CC >=12.0,<13.0
Please follow the instructions at https://pytorch.org/get-started/locally/ to install a PyTorch release that supports one of these CUDA versions: 12.6
  _warn_unsupported_code(d, device_cc, code_ccs)
/home/candimadam/Documents/Tugas Akhir/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:489: UserWarning: 
NVIDIA GeForce MX350 with CUDA capability sm_61 is not compatible with the curre

---
## 2. Load Dataset JSON

In [3]:
# Path file dataset
DATA_PATH = 'data_video.json'

# Load JSON
with open(DATA_PATH, 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

# Konversi ke DataFrame
df = pd.DataFrame(raw_data)

# Informasi Dataset
print('=' * 55)
print('           INFORMASI DATASET')
print('=' * 55)
print(f'Total record (video)   : {len(df):,}')
print(f'Total kolom            : {len(df.columns)}')
print(f'Nama kolom             : {list(df.columns)}')
print(f'Jumlah channel unik    : {df["nama_channel"].nunique()}')
print(f'Jumlah kategori unik   : {df["kategori"].nunique()}')
print('=' * 55)

# Distribusi video per channel
dist = df['nama_channel'].value_counts()
print(f'\nDistribusi jumlah video per channel:')
print(f'  Min  : {dist.min()} video')
print(f'  Max  : {dist.max()} video')
print(f'  Rata : {dist.mean():.1f} video')
print()

# Tampilkan 5 baris pertama
df.head()

           INFORMASI DATASET
Total record (video)   : 10,000
Total kolom            : 9
Nama kolom             : ['id', 'link_channel', 'nama_channel', 'kategori', 'jumlah_pelanggan', 'judul', 'link', 'jumlah_tayangan', 'tanggal_upload']
Jumlah channel unik    : 100
Jumlah kategori unik   : 10

Distribusi jumlah video per channel:
  Min  : 100 video
  Max  : 100 video
  Rata : 100.0 video



,id,link_channel,nama_channel,kategori,jumlah_pelanggan,judul,link,jumlah_tayangan,tanggal_upload
0,1,https://www.youtube.com/@GadgetIn/videos,@GadgetIn,Gadgets,13800000,Unboxing iPhone 17 Pro PALSU yang SANGAT MIRIP...,https://www.youtube.com/watch?v=zE5H9KQ_Hyg,1700000,5 days ago
1,2,https://www.youtube.com/@GadgetIn/videos,@GadgetIn,Gadgets,13800000,RAJA TERAKHIR HP SAMSUNG!,https://www.youtube.com/watch?v=snB4jbtscxU,932000,10 days ago
2,3,https://www.youtube.com/@GadgetIn/videos,@GadgetIn,Gadgets,13800000,Rp1.599 Juta! Ketika OPPO NIAT bikin HP murah...,https://www.youtube.com/watch?v=3KBJtbEAdzs,802000,11 days ago
3,4,https://www.youtube.com/@GadgetIn/videos,@GadgetIn,Gadgets,13800000,"Kalau Apple niat, iPhone bisa seworth it ini.....",https://www.youtube.com/watch?v=rkLpVyRGCPw,1300000,2 weeks ago
4,5,https://www.youtube.com/@GadgetIn/videos,@GadgetIn,Gadgets,13800000,Xiaomi pun ngeluh soal fenomena ini...,https://www.youtube.com/watch?v=Z4m_fwJ5eHQ&pp...,1200000,2 weeks ago


In [4]:
# Daftar channel yang tersedia 
print('Daftar Channel YouTube dalam Dataset:')
print('-' * 50)
for i, (ch, cnt) in enumerate(df['nama_channel'].value_counts().items(), 1):
    print(f'{i:>3}. {ch:<40} ({cnt} video)')

Daftar Channel YouTube dalam Dataset:
--------------------------------------------------
  1. @GadgetIn                                (100 video)
  2. @JagatReview                             (100 video)
  3. @GadgetGaul                              (100 video)
  4. @DHIARCOM                                (100 video)
  5. @DKIDchannel                             (100 video)
  6. @PricebookIndonesia                      (100 video)
  7. @Sobat_HAPE                              (100 video)
  8. @projectreview                           (100 video)
  9. @K2G                                     (100 video)
 10. @YoutuberCupu                            (100 video)
 11. @NexCarlos                               (100 video)
 12. @riasukmawijaya                          (100 video)
 13. @tanboykun                               (100 video)
 14. @KUBILER                                 (100 video)
 15. @MamankKuliner                           (100 video)
 16. @Melkibajaj                         

---
## 3. Text Cleaning

Pembersihan teks secara **minimalis**: hanya menghapus noise tanpa makna bahasa
(emoji, simbol dekoratif non-standar, karakter encoding rusak).
Tanda baca standar **(titik, koma, tanda tanya)** tetap dipertahankan karena IndoBERT memanfaatkannya.

    Tahapan:
    1. Menghapus emoji dan simbol Unicode non-standar
    2. Menghapus karakter encoding yang rusak / tidak dikenali
    3. Menghapus karakter khusus / dekoratif yang bukan tanda baca standar
    4. Merapikan spasi berlebih

In [5]:
def text_cleaning(text):
    if not isinstance(text, str):
        return ''

    # 1. Hapus emoji dan simbol Unicode non-standar
    emoji_pattern = re.compile(
        "["
        "\U0001F600-\U0001F64F"   # emoticons wajah
        "\U0001F300-\U0001F5FF"   # simbol & piktogram
        "\U0001F680-\U0001F6FF"   # transport & peta
        "\U0001F1E0-\U0001F1FF"   # bendera
        "\U00002600-\U000026FF"   # simbol umum
        "\U00002700-\U000027BF"   # Dingbats
        "\U0001F900-\U0001F9FF"   # simbol tambahan
        "\U0001FA00-\U0001FA6F"   # simbol tambahan-A
        "\U0001FA70-\U0001FAFF"   # simbol tambahan-B
        "\U00002300-\U000023FF"   # teknis
        "]+",
        flags=re.UNICODE
    )
    text = emoji_pattern.sub(' ', text)

    # 2. Hapus karakter non-printable / encoding rusak
    text = re.sub(r'[\x00-\x08\x0B-\x0C\x0E-\x1F\x7F]', ' ', text)

    # 3. Hapus simbol dekoratif yang bukan tanda baca standar
    #    Pertahankan: huruf, angka, spasi, . , ? ! - ( ) / @ # % + = : ; ' "
    text = re.sub(r'[^\w\s.,?!\-()/@#%+=\'":;]', ' ', text, flags=re.UNICODE)

    # 4. Rapikan spasi berlebih
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# Terapkan Text Cleaning
df['judul_clean'] = df['judul'].apply(text_cleaning)

# Tampilkan contoh hasil cleaning
print('Contoh Hasil Text Cleaning:')
print('=' * 70)
for _, row in df[['judul', 'judul_clean']].head(8).iterrows():
    print(f'SEBELUM : {row["judul"]}')
    print(f'SESUDAH : {row["judul_clean"]}')
    print('-' * 70)

Contoh Hasil Text Cleaning:
SEBELUM : Unboxing iPhone 17 Pro PALSU yang SANGAT MIRIP ASLINYA...
SESUDAH : Unboxing iPhone 17 Pro PALSU yang SANGAT MIRIP ASLINYA...
----------------------------------------------------------------------
SEBELUM : RAJA TERAKHIR HP SAMSUNG!
SESUDAH : RAJA TERAKHIR HP SAMSUNG!
----------------------------------------------------------------------
SEBELUM : Rp1.599 Juta! Ketika OPPO NIAT bikin HP murah...
SESUDAH : Rp1.599 Juta! Ketika OPPO NIAT bikin HP murah...
----------------------------------------------------------------------
SEBELUM : Kalau Apple niat, iPhone bisa seworth it ini... - Review iPhone 17
SESUDAH : Kalau Apple niat, iPhone bisa seworth it ini... - Review iPhone 17
----------------------------------------------------------------------
SEBELUM : Xiaomi pun ngeluh soal fenomena ini...
SESUDAH : Xiaomi pun ngeluh soal fenomena ini...
----------------------------------------------------------------------
SEBELUM : Rekomendasi HP TERBAIK buat A

---
## 4. Case Folding

Mengubah seluruh teks menjadi huruf kecil (**lowercase**) agar sesuai dengan
model **IndoBERT Base Uncased** yang dilatih pada teks huruf kecil.

In [6]:
def case_folding(text):
    if not isinstance(text, str):
        return ''
    return text.lower()

# Terapkan case folding setelah text cleaning
df['judul_lower'] = df['judul_clean'].apply(case_folding)

# Tampilkan contoh
print('Contoh Hasil Case Folding (setelah Text Cleaning):')
print('=' * 70)
for _, row in df[['judul_clean', 'judul_lower']].head(5).iterrows():
    print(f'SEBELUM : {row["judul_clean"]}')
    print(f'SESUDAH : {row["judul_lower"]}')
    print('-' * 70)

Contoh Hasil Case Folding (setelah Text Cleaning):
SEBELUM : Unboxing iPhone 17 Pro PALSU yang SANGAT MIRIP ASLINYA...
SESUDAH : unboxing iphone 17 pro palsu yang sangat mirip aslinya...
----------------------------------------------------------------------
SEBELUM : RAJA TERAKHIR HP SAMSUNG!
SESUDAH : raja terakhir hp samsung!
----------------------------------------------------------------------
SEBELUM : Rp1.599 Juta! Ketika OPPO NIAT bikin HP murah...
SESUDAH : rp1.599 juta! ketika oppo niat bikin hp murah...
----------------------------------------------------------------------
SEBELUM : Kalau Apple niat, iPhone bisa seworth it ini... - Review iPhone 17
SESUDAH : kalau apple niat, iphone bisa seworth it ini... - review iphone 17
----------------------------------------------------------------------
SEBELUM : Xiaomi pun ngeluh soal fenomena ini...
SESUDAH : xiaomi pun ngeluh soal fenomena ini...
----------------------------------------------------------------------


---
## 5. Tokenisasi dengan Stanza (Bahasa Indonesia)

Stanza digunakan untuk **tokenisasi** teks Bahasa Indonesia.
Output berupa teks yang sudah dinormalisasi (token yang bergabung kembali).

In [7]:
# Langkah 1: Download model Bahasa Indonesia
stanza.download('id', verbose=False)   # 'id' = kode bahasa Indonesia
print('Model Stanza Bahasa Indonesia siap.')

Model Stanza Bahasa Indonesia siap.


In [8]:
# Langkah 2: Inisialisasi pipeline Stanza 
# Hanya processor 'tokenize' yang diaktifkan untuk efisiensi

try:
    nlp_stanza = stanza.Pipeline(
        lang='id',
        processors='tokenize',
        verbose=False
    )
    print("✓ Stanza initialized on GPU")
except RuntimeError as e:
    print(f"⚠️  GPU error: {str(e)[:80]}...")
    print("   Fallback ke CPU...")
    nlp_stanza = stanza.Pipeline(
        lang='id',
        processors='tokenize',
        device='cpu',
        verbose=False
    )
    print("✓ Stanza initialized on CPU")

def tokenize_stanza(text):
    if not text or not text.strip():
        return text

    doc = nlp_stanza(text)
    # Gabungkan semua token dari semua kalimat
    tokens = []
    for sentence in doc.sentences:
        for token in sentence.tokens:
            tokens.append(token.text)

    return ' '.join(tokens)

# Langkah 3: Terapkan tokenisasi ke seluruh dataset
print(f'Memproses tokenisasi Stanza untuk {len(df):,} judul...')
print('(Proses ini memerlukan beberapa menit)\n')

tqdm.pandas(desc='Tokenisasi Stanza')
df['judul_tokenized'] = df['judul_lower'].progress_apply(tokenize_stanza)

print('\nTokenisasi selesai!')
print('\nContoh hasil tokenisasi:')
print('=' * 70)
for _, row in df[['judul_lower', 'judul_tokenized']].head(5).iterrows():
    print(f'INPUT  : {row["judul_lower"]}')
    print(f'OUTPUT : {row["judul_tokenized"]}')
    print('-' * 70)

⚠️  GPU error: cuDNN version 91900 is not compatible with devices with SM < 7.5. Please install...
   Fallback ke CPU...
✓ Stanza initialized on CPU
Memproses tokenisasi Stanza untuk 10,000 judul...
(Proses ini memerlukan beberapa menit)



Tokenisasi Stanza: 100%|██████████| 10000/10000 [02:05<00:00, 79.80it/s]


Tokenisasi selesai!

Contoh hasil tokenisasi:
INPUT  : unboxing iphone 17 pro palsu yang sangat mirip aslinya...
OUTPUT : unboxing iphone 17 pro palsu yang sangat mirip aslinya . . .
----------------------------------------------------------------------
INPUT  : raja terakhir hp samsung!
OUTPUT : raja terakhir hp samsung !
----------------------------------------------------------------------
INPUT  : rp1.599 juta! ketika oppo niat bikin hp murah...
OUTPUT : rp1.599 juta ! ketika oppo niat bikin hp murah . . .
----------------------------------------------------------------------
INPUT  : kalau apple niat, iphone bisa seworth it ini... - review iphone 17
OUTPUT : kalau apple niat , iphone bisa seworth it ini . . . - review iphone 17
----------------------------------------------------------------------
INPUT  : xiaomi pun ngeluh soal fenomena ini...
OUTPUT : xiaomi pun ngeluh soal fenomena ini . . .
----------------------------------------------------------------------


---
## 6. Split Data Terintegrasi 70/10/20 (Train/Validation/Test Channel)

Split utama mengikuti skema **70/10/20** di level channel:
- **Train channel (70%)**: untuk training fine-tuning dan kandidat rekomendasi
- **Validation channel (10%)**: khusus validasi fine-tuning
- **Test channel (20%)**: sebagai Query Set evaluasi rekomendasi

Untuk rekomendasi pada notebook ini:
- **Candidate Set** = Train channels (70%)
- **Query Set** = Test channels (20%)

In [9]:
from sklearn.model_selection import train_test_split

# ============================================================
# Split utama 70/10/20 di level channel (stratified by kategori)
# ============================================================
channel_level_df = (
    df.drop_duplicates(subset='nama_channel')[['nama_channel', 'kategori']]
      .reset_index(drop=True)
)

train_ch_df, temp_ch_df = train_test_split(
    channel_level_df,
    test_size=0.3,
    random_state=42,
    stratify=channel_level_df['kategori']
 )

val_ch_df, test_ch_df = train_test_split(
    temp_ch_df,
    test_size=2/3,  # 20% total test, 10% total validation
    random_state=42,
    stratify=temp_ch_df['kategori']
 )

train_channels = train_ch_df['nama_channel'].tolist()
val_channels = val_ch_df['nama_channel'].tolist()
test_channels = test_ch_df['nama_channel'].tolist()

# Untuk sistem rekomendasi mengikuti skema 70/10/20
candidate_channels = train_channels   # 70%
query_channels = test_channels        # 20%

candidate_df = df[df['nama_channel'].isin(candidate_channels)].copy()
query_df = df[df['nama_channel'].isin(query_channels)].copy()

# Untuk fine-tuning (supervised)
def build_supervised_split(channel_list):
    out = df[df['nama_channel'].isin(channel_list)][['judul_tokenized', 'kategori']].dropna().copy()
    out = out[out['judul_tokenized'].str.strip() != ''].reset_index(drop=True)
    return out

train_sup_df = build_supervised_split(train_channels)
val_sup_df = build_supervised_split(val_channels)
test_sup_df = build_supervised_split(test_channels)

supervised_pool = pd.concat([train_sup_df, val_sup_df, test_sup_df], ignore_index=True)
label_list = sorted(supervised_pool['kategori'].unique().tolist())
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for label, i in label2id.items()}

train_df = train_sup_df.copy()
val_df = val_sup_df.copy()
test_df = test_sup_df.copy()
train_df['label_id'] = train_df['kategori'].map(label2id)
val_df['label_id'] = val_df['kategori'].map(label2id)
test_df['label_id'] = test_df['kategori'].map(label2id)

print('=' * 85)
print('RINGKASAN SPLIT DATA 70/10/20 (LEVEL CHANNEL)')
print('=' * 85)
print(f'Total channel                    : {len(channel_level_df):,}')
print(f'Train channels (70%)             : {len(train_channels):,}')
print(f'Validation channels (10%)        : {len(val_channels):,}')
print(f'Test channels (20%)              : {len(test_channels):,}')
print('-' * 85)
print(f'Candidate channels (train only)  : {len(candidate_channels):,}')
print(f'Query channels (test)            : {len(query_channels):,}')
print('-' * 85)
print(f'Train videos (fine-tuning)       : {len(train_df):,}')
print(f'Validation videos (fine-tuning)  : {len(val_df):,}')
print(f'Test videos (held-out)           : {len(test_df):,}')
print(f'Jumlah label kategori            : {len(label_list)}')
print('=' * 85)

RINGKASAN SPLIT DATA 70/10/20 (LEVEL CHANNEL)
Total channel                    : 100
Train channels (70%)             : 70
Validation channels (10%)        : 10
Test channels (20%)              : 20
-------------------------------------------------------------------------------------
Candidate channels (train only)  : 70
Query channels (test)            : 20
-------------------------------------------------------------------------------------
Train videos (fine-tuning)       : 7,000
Validation videos (fine-tuning)  : 1,000
Test videos (held-out)           : 2,000
Jumlah label kategori            : 10


---
## 7. Konfigurasi Model IndoBERT

Model dasar yang digunakan untuk fine-tuning adalah `indolem/indobert-base-uncased`.

In [10]:
INDOBERT_MODEL_NAME = 'indolem/indobert-base-uncased'

print(f'Model dasar fine-tuning : {INDOBERT_MODEL_NAME}')
print(f'Device                 : {device}')

# Cek tokenizer saja di tahap ini (model akan di-load saat training fine-tuning)
tokenizer_preview = AutoTokenizer.from_pretrained(INDOBERT_MODEL_NAME)
print(f'Vocab size tokenizer   : {tokenizer_preview.vocab_size:,}')

Model dasar fine-tuning : indolem/indobert-base-uncased
Device                 : cpu


Vocab size tokenizer   : 31,923


---
## 8. Fine-Tuning IndoBERT (Supervised Classification)

Fine-tuning dilakukan pada data train dan dipantau menggunakan validation set.
Objective: klasifikasi kategori judul video menggunakan cross-entropy loss.

In [11]:
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModelForSequenceClassification, get_linear_schedule_with_warmup

class TextClassificationDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]

def collate_batch(batch, tokenizer, max_length=128):
    texts, labels = zip(*batch)
    enc = tokenizer(
        list(texts),
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors='pt'
    )
    labels = torch.tensor(labels, dtype=torch.long)
    return enc, labels

def evaluate_classifier(model, data_loader, device):
    model.eval()
    total = 0
    correct = 0

    with torch.no_grad():
        for batch_inputs, batch_labels in data_loader:
            batch_inputs = {k: v.to(device) for k, v in batch_inputs.items()}
            batch_labels = batch_labels.to(device)

            outputs = model(**batch_inputs)
            preds = outputs.logits.argmax(dim=-1)

            total += batch_labels.size(0)
            correct += (preds == batch_labels).sum().item()

    return correct / max(total, 1)

def train_classifier_fine_tune(train_df, val_df, model_name='indolem/indobert-base-uncased',
                               num_epochs=2, batch_size=16, lr=2e-5, max_length=128):
    """Fine-tune IndoBERT untuk klasifikasi kategori judul video."""
    tokenizer_local = AutoTokenizer.from_pretrained(model_name)
    num_labels = len(label_list)

    clf_model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=num_labels,
        id2label=id2label,
        label2id=label2id
    )
    clf_model = clf_model.to(device)

    train_dataset = TextClassificationDataset(
        train_df['judul_tokenized'].tolist(),
        train_df['label_id'].tolist()
    )
    val_dataset = TextClassificationDataset(
        val_df['judul_tokenized'].tolist(),
        val_df['label_id'].tolist()
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=lambda b: collate_batch(b, tokenizer_local, max_length=max_length)
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=lambda b: collate_batch(b, tokenizer_local, max_length=max_length)
    )

    optimizer = torch.optim.AdamW(clf_model.parameters(), lr=lr)
    total_steps = max(len(train_loader) * num_epochs, 1)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(0.1 * total_steps),
        num_training_steps=total_steps
    )

    history = []
    for epoch in range(1, num_epochs + 1):
        clf_model.train()
        running_loss = 0.0

        for batch_inputs, batch_labels in tqdm(train_loader, desc=f'Fine-tune epoch {epoch}/{num_epochs}', unit='batch'):
            batch_inputs = {k: v.to(device) for k, v in batch_inputs.items()}
            batch_labels = batch_labels.to(device)

            outputs = clf_model(**batch_inputs, labels=batch_labels)
            loss = outputs.loss

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            scheduler.step()

            running_loss += loss.item()

        avg_train_loss = running_loss / max(len(train_loader), 1)
        val_acc = evaluate_classifier(clf_model, val_loader, device)
        history.append({'epoch': epoch, 'train_loss': round(avg_train_loss, 4), 'val_acc': round(val_acc, 4)})
        print(f'  Epoch {epoch}/{num_epochs} | train_loss={avg_train_loss:.4f} | val_acc={val_acc:.4f}')

    return {
        'model': clf_model,
        'tokenizer': tokenizer_local,
        'history': pd.DataFrame(history),
        'best_val_acc': max(h['val_acc'] for h in history) if history else 0.0
    }

NUM_EPOCHS = 2
BATCH_SIZE = 16
LEARNING_RATE = 2e-5
MAX_LENGTH = 128

print('=' * 80)
print('FINE-TUNING INDOBERT (PRETRAINED -> FINE-TUNED)')
print('=' * 80)
print(f'Dataset          : {len(train_df):,} train, {len(val_df):,} val')
print(f'Labels           : {len(label_list)} kategori')
print(f'Epochs           : {NUM_EPOCHS}')
print(f'Batch size       : {BATCH_SIZE}')
print(f'Learning rate    : {LEARNING_RATE}')
print(f'Device           : {device}')
print('=' * 80)

fine_tuned_result = train_classifier_fine_tune(
    train_df=train_df,
    val_df=val_df,
    num_epochs=NUM_EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LEARNING_RATE,
    max_length=MAX_LENGTH
)

print('\nFine-tuning selesai.')
print(f'Best validation accuracy: {fine_tuned_result["best_val_acc"]:.4f}')

FINE-TUNING INDOBERT (PRETRAINED -> FINE-TUNED)
Dataset          : 7,000 train, 1,000 val
Labels           : 10 kategori
Epochs           : 2
Batch size       : 16
Learning rate    : 2e-05
Device           : cpu


[transformers] You passed `num_labels=10` which is incompatible to the `id2label` map of length `2`.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 10525.69it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: indolem/indobert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not

  Epoch 1/2 | train_loss=1.4742 | val_acc=0.6720


Fine-tune epoch 2/2: 100%|██████████| 438/438 [24:12<00:00,  3.32s/batch]


  Epoch 2/2 | train_loss=0.5242 | val_acc=0.7190

Fine-tuning selesai.
Best validation accuracy: 0.7190


---
## 9. Generate Embedding Judul Video (Fine-Tuned Encoder)

Embedding dibuat menggunakan encoder hasil fine-tuning dengan metode masked mean pooling.

In [12]:
def generate_finetuned_embeddings(texts, tokenizer_local, encoder_model, device, batch_size=32, max_length=128):
    encoder_model.eval()
    all_emb = []

    for i in tqdm(range(0, len(texts), batch_size), desc='Generating fine-tuned embeddings', unit='batch'):
        batch_texts = texts[i:i + batch_size]
        inputs = tokenizer_local(
            batch_texts,
            return_tensors='pt',
            truncation=True,
            padding=True,
            max_length=max_length
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            out = encoder_model(**inputs).last_hidden_state

        mask = inputs['attention_mask'].unsqueeze(-1).float()
        summed = (out * mask).sum(dim=1)
        counts = torch.clamp(mask.sum(dim=1), min=1e-9)
        pooled = summed / counts
        all_emb.append(pooled.cpu().numpy())

    return np.vstack(all_emb)

texts_to_embed = df['judul_tokenized'].tolist()
ft_encoder = fine_tuned_result['model'].base_model
ft_tokenizer = fine_tuned_result['tokenizer']

print('=' * 80)
print('GENERATE EMBEDDING DENGAN MODEL FINE-TUNED')
print('=' * 80)
ft_video_embeddings = generate_finetuned_embeddings(
    texts=texts_to_embed,
    tokenizer_local=ft_tokenizer,
    encoder_model=ft_encoder,
    device=device,
    batch_size=32,
    max_length=MAX_LENGTH
)

df_ft = df.copy()
df_ft['embedding'] = list(ft_video_embeddings)

print(f'Jumlah embedding: {ft_video_embeddings.shape[0]:,}')
print(f'Dimensi vektor  : {ft_video_embeddings.shape[1]}')

GENERATE EMBEDDING DENGAN MODEL FINE-TUNED


Generating fine-tuned embeddings: 100%|██████████| 313/313 [09:26<00:00,  1.81s/batch]

Jumlah embedding: 10,000
Dimensi vektor  : 768


---
## 10. Agregasi Vektor Channel (Mean Pooling + L2 Normalization)

Setiap channel direpresentasikan sebagai rata-rata embedding seluruh videonya.

In [13]:
from sklearn.preprocessing import normalize

def aggregate_channel_embeddings(input_df):
    grouped = input_df.groupby('nama_channel')
    return {
        ch: np.mean(np.stack(group['embedding'].values), axis=0)
        for ch, group in tqdm(grouped, desc='Agregasi channel', unit='channel')
    }

channel_vectors_ft = aggregate_channel_embeddings(df_ft)
channel_names_ft = list(channel_vectors_ft.keys())
ft_channel_matrix = np.stack([channel_vectors_ft[ch] for ch in channel_names_ft])
ft_channel_matrix = normalize(ft_channel_matrix, norm='l2')

print('=' * 80)
print('AGREGASI CHANNEL FINE-TUNED')
print('=' * 80)
print(f'Jumlah channel : {len(channel_names_ft)}')
print(f'Shape matrix   : {ft_channel_matrix.shape}')

Agregasi channel: 100%|██████████| 100/100 [00:00<00:00, 1548.24channel/s]

AGREGASI CHANNEL FINE-TUNED
Jumlah channel : 100
Shape matrix   : (100, 768)


---
## 11. Cosine Similarity Matrix & Fungsi Rekomendasi

Menghitung similarity antar channel dan menampilkan rekomendasi Top-K.

In [14]:
ft_similarity_matrix = cosine_similarity(ft_channel_matrix)
ft_similarity_df = pd.DataFrame(
    ft_similarity_matrix,
    index=channel_names_ft,
    columns=channel_names_ft
)

def recommend_channel(channel_name, top_k, similarity_df, channel_info, candidate_channels):
    sim_scores = similarity_df.loc[channel_name].drop(labels=channel_name)
    valid_candidates = [ch for ch in candidate_channels if ch in sim_scores.index]
    sim_scores = sim_scores.loc[valid_candidates]
    top_k_channels = sim_scores.sort_values(ascending=False).head(top_k)

    rows = []
    for rank, (ch, score) in enumerate(top_k_channels.items(), start=1):
        info = channel_info.loc[ch] if ch in channel_info.index else {'kategori': '-', 'jumlah_pelanggan': 0}
        rows.append({
            'rank': rank,
            'nama_channel': ch,
            'kategori': info['kategori'],
            'jumlah_pelanggan': info['jumlah_pelanggan'],
            'similarity_score': round(float(score), 4)
        })
    return pd.DataFrame(rows).set_index('rank')

channel_info = (
    df.drop_duplicates(subset='nama_channel')
      .set_index('nama_channel')[['kategori', 'jumlah_pelanggan']]
 )

input_channel = query_channels[0]
TOP_K = 5
rec_df = recommend_channel(
    channel_name=input_channel,
    top_k=TOP_K,
    similarity_df=ft_similarity_df,
    channel_info=channel_info,
    candidate_channels=candidate_channels
)

print('=' * 80)
print('CONTOH REKOMENDASI (FINE-TUNED)')
print('=' * 80)
print(f'Input channel: {input_channel}')
print(rec_df.to_string())

CONTOH REKOMENDASI (FINE-TUNED)
Input channel: @metrotvnews
            nama_channel kategori  jumlah_pelanggan  similarity_score
rank                                                                 
1              @kompastv     News          19700000            0.9353
2     @tempovideochannel     News           1880000            0.9336
3               @CNBC_ID     News           2900000            0.9312
4         @liputan6_news     News           2670000            0.9271
5            @tribunnews     News          15400000            0.9205


---
## 12. Evaluasi Sistem (Precision@K pada Query Set/Test Channel)

Evaluasi dilakukan pada **test channels (20%)** dengan definisi relevansi kategori yang sama.
Candidate rekomendasi hanya berasal dari **train channels (70%)**.

In [15]:
def precision_at_k(channel_name, k, similarity_df, channel_category_map, candidate_channels):
    sim_scores = similarity_df.loc[channel_name].drop(labels=channel_name)
    valid_candidates = [ch for ch in candidate_channels if ch in sim_scores.index]
    sim_scores = sim_scores.loc[valid_candidates]
    top_k_channels = sim_scores.sort_values(ascending=False).head(k).index.tolist()

    target_category = channel_category_map.get(channel_name)
    relevant_count = sum(channel_category_map.get(ch) == target_category for ch in top_k_channels)
    return relevant_count / k

def evaluate_system(k_values, eval_channels, similarity_df, channel_category_map, candidate_channels):
    rows = []
    for ch in tqdm(eval_channels, desc='Evaluasi Precision@K', unit='channel'):
        row = {'channel': ch, 'kategori': channel_category_map.get(ch, '-') }
        for k in k_values:
            row[f'P@{k}'] = round(
                precision_at_k(ch, k, similarity_df, channel_category_map, candidate_channels),
                4
            )
        rows.append(row)

    eval_out = pd.DataFrame(rows).set_index('channel')
    avg_row = {'kategori': 'AVERAGE'}
    for k in k_values:
        avg_row[f'P@{k}'] = round(eval_out[f'P@{k}'].mean(), 4)

    return pd.concat([eval_out, pd.DataFrame([avg_row], index=['--- AVERAGE ---'])])

channel_category_map = (
    df.drop_duplicates(subset='nama_channel')
      .set_index('nama_channel')['kategori']
      .to_dict()
)

K_VALUES = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
ft_eval_df = evaluate_system(
    k_values=K_VALUES,
    eval_channels=query_channels,
    similarity_df=ft_similarity_df,
    channel_category_map=channel_category_map,
    candidate_channels=candidate_channels
)

print('=' * 80)
print('HASIL EVALUASI FINE-TUNED (Precision@K - Query Set)')
print('=' * 80)
print(ft_eval_df.to_string())

Evaluasi Precision@K: 100%|██████████| 20/20 [00:00<00:00, 62.82channel/s]

HASIL EVALUASI FINE-TUNED (Precision@K - Query Set)
                         kategori  P@1  P@2     P@3     P@4   P@5    P@6     P@7    P@8     P@9   P@10
@metrotvnews                 News  1.0  1.0  1.0000  1.0000  1.00  1.000  1.0000  0.875  0.7778  0.700
@Audrey-A                 Animals  1.0  1.0  1.0000  1.0000  1.00  1.000  1.0000  0.875  0.7778  0.700
@BimbelBrilian          Education  1.0  1.0  1.0000  1.0000  1.00  1.000  0.8571  0.750  0.6667  0.600
@LuckyHakimChannel        Animals  1.0  1.0  1.0000  1.0000  1.00  1.000  1.0000  0.875  0.7778  0.700
@GadgetIn                 Gadgets  1.0  1.0  1.0000  1.0000  1.00  1.000  1.0000  0.875  0.7778  0.700
@AttaHalilintar     Entertainment  1.0  1.0  1.0000  1.0000  1.00  1.000  1.0000  0.875  0.7778  0.700
@ZeniusEducation        Education  1.0  1.0  1.0000  1.0000  1.00  1.000  0.8571  0.750  0.7778  0.700
@AfifYulistian             Gaming  1.0  1.0  1.0000  1.0000  1.00  1.000  1.0000  0.875  0.7778  0.700
@Anak.Kuliner        

In [16]:
# Ringkasan Rata-Rata Precision@K untuk Query Set
print('\nRingkasan Rata-Rata Precision@K untuk Query Set:')
avg_series = ft_eval_df.loc['--- AVERAGE ---']
for k in K_VALUES:
    print(f'  Precision@{k:>2} = {avg_series[f"P@{k}"]:.4f}')

# Precision@K per kategori
print('\n\nPrecision@K Per Kategori (rata-rata per kategori):')
print('=' * 55)

# Ambil hanya baris channel (hapus baris AVERAGE)
ft_eval_df_clean = ft_eval_df[ft_eval_df['kategori'] != 'AVERAGE'].copy()
pk_cols = [f'P@{k}' for k in K_VALUES]

cat_summary = (
    ft_eval_df_clean.groupby('kategori')[pk_cols]
    .mean()
    .round(4)
    .sort_values('P@5', ascending=False)
 )

# Top 10 kategori terbaik berdasarkan P@5
top10_cat = cat_summary['P@5'].head(10)

print('\nTop 10 Kategori dengan Kemiripan Tertinggi (berdasarkan P@5):')
for i, (kategori, score) in enumerate(top10_cat.items(), start=1):
    print(f'{i:>2}. {kategori:<20} -> P@5 = {score:.4f}')

# Jika perlu lihat tabel lengkap per kategori, uncomment baris di bawah
print('\n'); print(cat_summary.to_string())


Ringkasan Rata-Rata Precision@K untuk Query Set:
  Precision@ 1 = 0.9000
  Precision@ 2 = 0.9000
  Precision@ 3 = 0.9167
  Precision@ 4 = 0.9125
  Precision@ 5 = 0.9200
  Precision@ 6 = 0.9250
  Precision@ 7 = 0.9143
  Precision@ 8 = 0.8000
  Precision@ 9 = 0.7222
  Precision@10 = 0.6550


Precision@K Per Kategori (rata-rata per kategori):

Top 10 Kategori dengan Kemiripan Tertinggi (berdasarkan P@5):
 1. Animals              -> P@5 = 1.0000
 2. Automotive           -> P@5 = 1.0000
 3. Education            -> P@5 = 1.0000
 4. Food                 -> P@5 = 1.0000
 5. Gadgets              -> P@5 = 1.0000
 6. Gaming               -> P@5 = 1.0000
 7. Sports               -> P@5 = 1.0000
 8. News                 -> P@5 = 1.0000
 9. Entertainment        -> P@5 = 0.7000
10. Music                -> P@5 = 0.5000


               P@1  P@2     P@3    P@4  P@5   P@6     P@7     P@8     P@9  P@10
kategori                                                                       
Animals        1.0  1.

---
## 13. Simpan Artefak Final Fine-Tuning

Semua artefak final disimpan ke folder `output/fine_tuning_experiment/`.

In [17]:
OUTPUT_DIR = 'output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

FT_OUTPUT_DIR = os.path.join(OUTPUT_DIR, 'fine_tuning_experiment')
os.makedirs(FT_OUTPUT_DIR, exist_ok=True)

print('=' * 80)
print(f'Menyimpan artefak fine-tuning ke: {FT_OUTPUT_DIR}')
print('=' * 80)

# 1) Split fine-tuning
train_df.to_csv(os.path.join(FT_OUTPUT_DIR, 'train_split.csv'), index=False)
val_df.to_csv(os.path.join(FT_OUTPUT_DIR, 'val_split.csv'), index=False)

# 2) Riwayat training
fine_tuned_result['history'].to_csv(os.path.join(FT_OUTPUT_DIR, 'training_history.csv'), index=False)

# 3) Hasil evaluasi rekomendasi
ft_eval_df.to_csv(os.path.join(FT_OUTPUT_DIR, 'evaluation_precision_at_k.csv'))

# 4) Matrix hasil fine-tuned
ft_similarity_df.to_csv(os.path.join(FT_OUTPUT_DIR, 'similarity_matrix_fine_tuned.csv'))
np.save(os.path.join(FT_OUTPUT_DIR, 'channel_matrix_fine_tuned.npy'), ft_channel_matrix)
np.save(os.path.join(FT_OUTPUT_DIR, 'video_embeddings_fine_tuned.npy'), ft_video_embeddings)

# 5) Model fine-tuned
fine_tuned_result['model'].save_pretrained(os.path.join(FT_OUTPUT_DIR, 'model_fine_tuned'))
fine_tuned_result['tokenizer'].save_pretrained(os.path.join(FT_OUTPUT_DIR, 'model_fine_tuned'))

# 6) Metadata
metadata = {
    'model_name': INDOBERT_MODEL_NAME,
    'best_validation_accuracy': float(fine_tuned_result['best_val_acc']),
    'num_epochs': NUM_EPOCHS,
    'batch_size': BATCH_SIZE,
    'learning_rate': LEARNING_RATE,
    'max_length': MAX_LENGTH,
    'k_values': K_VALUES,
    'candidate_channels_count': len(candidate_channels),
    'query_channels_count': len(query_channels),
    'num_labels': len(label_list),
    'labels': label_list
}
with open(os.path.join(FT_OUTPUT_DIR, 'metadata.json'), 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print('\nArtefak tersimpan:')
for fname in sorted(os.listdir(FT_OUTPUT_DIR)):
    fpath = os.path.join(FT_OUTPUT_DIR, fname)
    print(f'  - {fname}/' if os.path.isdir(fpath) else f'  - {fname}')

print('\nSelesai menyimpan artefak fine-tuning.')

Menyimpan artefak fine-tuning ke: output/fine_tuning_experiment


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.63it/s]


Artefak tersimpan:
  - channel_matrix_fine_tuned.npy
  - evaluation_precision_at_k.csv
  - metadata.json
  - model_fine_tuned/
  - similarity_matrix_fine_tuned.csv
  - train_split.csv
  - training_history.csv
  - val_split.csv
  - video_embeddings_fine_tuned.npy

Selesai menyimpan artefak fine-tuning.


---
## Ringkasan Metodologi (Fine-Tuning Only)

| No | Tahap | Metode | Catatan |
|---|---|---|---|
| 1 | Install & Import Library | Setup dependency dan import modul utama | `torch`, `transformers`, `stanza`, `sklearn`, `pandas`, `numpy`, `tqdm` |
| 2 | Load Dataset JSON | Membaca data dan membentuk DataFrame | `json`, `pandas` |
| 3 | Text Cleaning | Hapus emoji, simbol non-standar, encoding rusak | `re` |
| 4 | Case Folding | Lowercase seluruh teks | built-in |
| 5 | Tokenisasi dengan Stanza | Tokenisasi Bahasa Indonesia | `stanza` |
| 6 | Split Data Utama | Train/Validation/Test = 70/10/20 (level channel) | Candidate = train, Query = test |
| 7 | Konfigurasi Model | Siapkan model dasar IndoBERT | `indolem/indobert-base-uncased` |
| 8 | Fine-Tuning | Klasifikasi kategori judul video | cross-entropy, AdamW, warmup scheduler |
| 9 | Embedding Fine-Tuned | Masked mean pooling dari encoder hasil fine-tuning | `torch`, `numpy` |
| 10 | Agregasi Vektor Channel | Mean pooling antar video per channel + L2 normalization | `numpy`, `sklearn` |
| 11 | Similarity & Rekomendasi | Cosine similarity dan Top-K rekomendasi | `pandas`, `numpy` |
| 12 | Evaluasi | Precision@K pada Query Set (test channel) | ground truth: kategori sama |
| 13 | Simpan Artefak | Model, matrix, dan hasil evaluasi final | `os`, `json`, `pandas`, `numpy` |

**Pendekatan:** Content-Based Filtering dengan Fine-Tuned Embedding  
**Model NLP:** `indolem/indobert-base-uncased` (fine-tuned pada train set)  
**Fitur Utama:** Judul Video YouTube